In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(
    model=model,
    mu=1e-3,
    lr_init=1e-3,
    damping="identity",
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.532154381275177
epoch:  1, loss: 0.5244811773300171
epoch:  2, loss: 0.5170244574546814
epoch:  3, loss: 0.5097542405128479
epoch:  4, loss: 0.5026135444641113
epoch:  5, loss: 0.49556732177734375
epoch:  6, loss: 0.48857611417770386
epoch:  7, loss: 0.48163601756095886
epoch:  8, loss: 0.47473379969596863
epoch:  9, loss: 0.46784332394599915
epoch:  10, loss: 0.4609462320804596
epoch:  11, loss: 0.4540339708328247
epoch:  12, loss: 0.4471432864665985
epoch:  13, loss: 0.4402875602245331
epoch:  14, loss: 0.43347182869911194
epoch:  15, loss: 0.42669838666915894
epoch:  16, loss: 0.41996896266937256
epoch:  17, loss: 0.4132925271987915
epoch:  18, loss: 0.40667369961738586
epoch:  19, loss: 0.40011435747146606
epoch:  20, loss: 0.3936183452606201
epoch:  21, loss: 0.3871927559375763
epoch:  22, loss: 0.38083958625793457
epoch:  23, loss: 0.3745606243610382
epoch:  24, loss: 0.368355929851532
epoch:  25, loss: 0.36222031712532043
epoch:  26, loss: 0.3561598062515259
e

KeyboardInterrupt: 

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.5150478836853787
Test metrics:  R2 = 0.5544333476040876


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    mu=1e-3,
    damping="identity",
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05146589130163193
epoch:  1, loss: 0.031186796724796295
epoch:  2, loss: 0.029248544946312904
epoch:  3, loss: 0.02579074539244175
epoch:  4, loss: 0.018820064142346382
epoch:  5, loss: 0.014280158095061779
epoch:  6, loss: 0.01203904952853918
epoch:  7, loss: 0.010780220851302147
epoch:  8, loss: 0.009876552037894726
epoch:  9, loss: 0.009260875172913074
epoch:  10, loss: 0.008540790528059006
epoch:  11, loss: 0.00685081584379077
epoch:  12, loss: 0.006221313029527664
epoch:  13, loss: 0.005732460413128138
epoch:  14, loss: 0.0053066955879330635


KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9873690009117126
Test metrics:  R2 = 0.8275281190872192
